Be sure to run the following notebook first before running this notebook:
- 1-load-and-convert-statsbomb-data.ipynb

In [1]:
import os
import tqdm
import pandas as pd
import numpy as np

In [2]:
%load_ext autoreload
%autoreload 2
import socceraction.spadl as spadl
import socceraction.vaep.features as fs
import socceraction.xthreat as xthreat

## Select data

In [3]:
# Configure file and folder names, use SPADL format.
datafolder = "../data"
spadl_h5 = os.path.join(datafolder, "spadl-statsbomb.h5")
xT_h5 = os.path.join(datafolder, "xT.h5")

In [4]:
games = pd.read_hdf(spadl_h5, "games")
print("nb of games:", len(games))

nb of games: 64


In [5]:
# Read in all actions of games
A = []

with pd.HDFStore(spadl_h5) as spadlstore:
    for game in tqdm.tqdm(list(games.itertuples())):
        actions = spadlstore[f"actions/game_{game.game_id}"]
        actions = spadl.add_names(actions)
        actions = spadl.play_left_to_right(actions, game.home_team_id)
        A.append(actions) 

A = pd.concat(A)

100%|██████████| 64/64 [00:01<00:00, 61.10it/s]


## Load pre-trained model

In [6]:
# uncomment the lines below if you get an SSLError
# import ssl
# ssl._create_default_https_context = ssl._create_unverified_context

url_grid = "https://karun.in/blog/data/open_xt_12x8_v1.json"
xTModel = xthreat.load_model(url_grid)

## Train a custom model

In [7]:
xTModel = xthreat.ExpectedThreat(l=16, w=12)
xTModel.fit(A);

# iterations:  37


## Compute xT ratings

In [8]:
## Predict

# xT should only be used to value actions that move the ball 
# and also keep the current team in possession of the ball
mov_actions = xthreat.get_successful_move_actions(A)
mov_actions["xT_value"] = xTModel.rate(mov_actions)
mov_actions[["type_name", "start_x", "start_y", "end_x", "end_y", "xT_value"]][:10]

,type_name,start_x,start_y,end_x,end_y,xT_value
0,pass,52.9375,34.340,44.6250,35.700,-0.001926
1,dribble,44.6250,35.700,43.8375,35.700,0.000000
2,pass,43.8375,35.700,26.7750,29.070,-0.001576
3,dribble,26.7750,29.070,30.0125,22.525,-0.000022
6,dribble,70.3500,17.340,59.9375,22.440,-0.001725
7,pass,59.9375,22.440,34.3000,28.390,-0.004525
8,dribble,34.3000,28.390,32.3750,29.665,-0.000821
9,pass,32.3750,29.665,10.3250,36.720,-0.001169
10,dribble,10.3250,36.720,13.4750,37.910,0.000276
11,pass,13.4750,37.910,18.1125,46.070,-0.000134


## Inspect the learned xT Model
Extra libraries required: matplotsoccer & plotly

In [12]:
xt_grid_numpy = xTModel.xT
df_xt_grid = pd.DataFrame(xt_grid_numpy)

output_csv_filename = "../Evaluation/wwc2023_trained_xT_grid.csv"

df_xt_grid.to_csv(output_csv_filename, index=False, header=False)